In [5]:
import numpy as np
import pandas as pd

np.random.seed(42)

# ===== Load citizen trust data =====
citizens = pd.read_csv("F:/GraduationDataset/1-Citizen Trust Score Model/citizen_trust_dataset.csv")

# لو لسه ما طلعتش trust_score من مودل
# استخدم valid_ratio كبديل مؤقت
if "trust_score" not in citizens.columns:
    citizens["trust_score"] = citizens["valid_ratio"]

citizen_ids = citizens["citizen_id"].values

# ===== Helpers =====
incident_types = ["Electricity", "Water", "Road", "Gas", "Waste"]
area_types = ["Residential", "Commercial", "Industrial"]

def generate_severity_label(score):
    if score < 0.4:
        return "Low"
    elif score < 0.7:
        return "Medium"
    else:
        return "High"

# ===== Generate Incidents =====
data = []

for incident_id in range(1, 30001):
    citizen = citizens.sample(1).iloc[0]

    trust = citizen["trust_score"]

    incident_type = np.random.choice(incident_types)
    area_type = np.random.choice(area_types)

    hour = np.random.randint(0, 24)
    day_of_week = np.random.randint(0, 7)

    anomaly_score = round(np.random.beta(2, 5), 2)
    image_auth_score = round(np.random.beta(5, 2), 2)

    reports_nearby_1h = np.random.poisson(lam=3)

    historical_severity_avg = round(np.random.uniform(1.5, 3.5), 2)

    # ===== Severity Logic =====
    severity_numeric = (
        0.35 * trust +
        0.25 * anomaly_score +
        0.25 * image_auth_score +
        0.15 * min(reports_nearby_1h / 10, 1)
    )

    severity_label = generate_severity_label(severity_numeric)

    data.append({
        "incident_id": incident_id,
        "citizen_id": citizen["citizen_id"],
        "incident_type": incident_type,
        "area_type": area_type,
        "hour": hour,
        "day_of_week": day_of_week,
        "citizen_trust_score": round(trust, 2),
        "anomaly_score": anomaly_score,
        "image_authenticity_score": image_auth_score,
        "reports_nearby_1h": reports_nearby_1h,
        "historical_severity_avg": historical_severity_avg,
        "severity_label": severity_label
    })

df_incidents = pd.DataFrame(data)

# ===== Save =====
df_incidents.to_csv("incident_severity_dataset.csv", index=False)

print("✅ Incident dataset generated:", df_incidents.shape)
print(df_incidents.head())


✅ Incident dataset generated: (30000, 12)
   incident_id  citizen_id incident_type   area_type  hour  day_of_week  \
0            1     10651.0         Waste  Industrial     1            0   
1            2      5502.0   Electricity  Commercial    15            2   
2            3      3210.0         Waste  Industrial    10            3   
3            4     12061.0           Gas  Commercial    22            6   
4            5     11319.0   Electricity  Commercial    10            6   

   citizen_trust_score  anomaly_score  image_authenticity_score  \
0                 0.70           0.32                      0.56   
1                 0.85           0.19                      0.40   
2                 0.87           0.42                      0.40   
3                 0.62           0.12                      0.85   
4                 0.60           0.40                      0.88   

   reports_nearby_1h  historical_severity_avg severity_label  
0                  3                     